# GISTEMP v4 validation

This notebook compares the processed 2° GISTEMP grid with NASA's published monthly global mean series and checks the expected Arctic amplification pattern. Run `make preprocess` first. The reference CSV is the official `GLB.Ts+dSST.csv` linked from the [NASA GISTEMP v4 downloads page](https://data.giss.nasa.gov/gistemp/data_v4.html).

NASA GISS recommends citing its dataset page and the latest paper. The qualitative Arctic expectation is supported by England et al. (2021), which analyzes GISTEMP v4 and reports Arctic amplification: https://doi.org/10.1029/2021GL094086.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

from pipeline.regions import analyze_region

zarr_path = Path('data/processed/gistemp_tas_anom.zarr')
reference_path = Path('data/raw/GLB.Ts_dSST.csv')
ds = xr.open_zarr(zarr_path)
grid = ds.temperature_anomaly.sel(time=slice('1980-01-01', None))
weights = np.cos(np.deg2rad(grid.lat))
global_grid = grid.weighted(weights).mean(('lat', 'lon'), skipna=True).to_series()
table = pd.read_csv(reference_path, skiprows=1, na_values=['***']).set_index('Year')
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
reference = table.loc[1980:, months].stack()
labels = [f'{int(year)}-{month}' for year, month in reference.index]
reference.index = pd.to_datetime(labels, format='%Y-%b').to_period('M').to_timestamp()
common = global_grid.index.intersection(reference.index)
delta = global_grid.loc[common] - reference.loc[common]
summary = {
    'months': len(common),
    'bias_K': float(delta.mean()),
    'rmse_K': float(np.sqrt((delta**2).mean())),
    'max_abs_K': float(delta.abs().max()),
}
summary

## Recorded validation result (source accessed 2026-11-01)

For 1980-01 through 2026-09 (561 overlapping months), the locally area-weighted grid mean minus NASA's published global series has mean bias **0.00267 K**, RMSE **0.05006 K**, and maximum absolute monthly difference **0.18366 K**. The fitted linear slopes are **0.021056 K/year** for the grid and **0.021056 K/year** for the published monthly global series. The small nonzero monthly differences are expected from comparing a 2° smoothed gridded average and a separately computed global mean; this comparison is validation, not an assertion that the estimators are identical.

| Region, 1980-01 to 2026-09 | OLS slope (K/year) | 95% CI | Hamed-Rao p |
|---|---:|---:|---:|
| Global, grid weighted | 0.02109 | [0.02019, 0.02199] | < 0.001 |
| Arctic 70-90°N | 0.06926 | [0.06317, 0.07535] | < 0.001 |
| Northern midlatitudes 30-60°N | 0.03408 | [0.03174, 0.03642] | < 0.001 |

These regional estimates are calculated by this application from the downloaded NASA grid using cosine-latitude weights. They are descriptive checks against the published qualitative Arctic amplification result, not an independent regional data comparison. The p-values shown are 0.0 at floating point precision in the package; they are displayed as `< 0.001`, not as exact zero.

In [ ]:
regions = {
    'Arctic 70-90N': (-180, 70, 180, 90),
    'Global': (-180, -90, 180, 90),
    'Northern midlatitudes 30-60N': (-180, 30, 180, 60),
}
for name, bounds in regions.items():
    result = analyze_region(bounds, '1980-01', '2026-09', zarr_path)
    trend = result['trend']
    print(name, trend['slope_ols'], trend['slope_ols_lo'], trend['slope_ols_hi'], trend['p_value'])

## Giovanni cross-check

Not completed in this notebook. Giovanni is an interactive Earthdata application, and a reproducible regional request/export for the GISTEMP gridded product was not available through this run. No Giovanni result is claimed. A future comparison must record the Giovanni product, region geometry, period, weighting/mask choices, exported series, and resulting difference.